<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Game

### Perparation

In [15]:
from typing import Any, Callable

class Game:
    def start_state(self) -> Any:
        """Where the game starts."""
        raise NotImplementedError

    def successors(self, state: Any) -> dict[str, Any]:
        """What are the possible successor states?"""
        raise NotImplementedError
    def player(self, state: Any) -> str:
        """Which player should move in `state`?"""
        raise NotImplementedError

    def is_end(self, state: Any) -> bool:
        """Is the game over?"""
        raise NotImplementedError

    def utility(self, state: Any) -> float:
        """What is the utility of the game (for the agent)."""
        raise NotImplementedError

## Game Modeling

### Ball Game

In [16]:
class Game1(Game):
    def start_state(self) -> Any:
        return "root"
    def successors(self, state: str) -> dict[str, str]:
        # state -> action -> next state
        mapping = {
            "root": {"A": "A", "B": "B", "C": "C"},
            "A": {"1": "A1", "2": "A2"},
            "B": {"1": "B1", "2": "B2"},
            "C": {"1": "C1", "2": "C2"},
        }
        return mapping[state]
    def player(self, state: Any) -> str:
        if state == "root":
            return "agent"
        if state in ["A", "B", "C"]:
            return "opp"
        raise ValueError(f"Invalid state: {state}")
    def is_end(self, state: Any) -> bool:
        return state in ["A1", "A2", "B1", "B2", "C1", "C2"]
    def utility(self, state: Any) -> float:
        utilities = {
            "A1": -50,
            "A2": 50,
            "B1": 1,
            "B2": 3,
            "C1": -5,
            "C2": 15,
        }
        return utilities[state]

In [17]:
game = Game1()

# From the root
state = game.start_state()  # Root node
print(f"Start State: {state}")

is_end = game.is_end(state)  # Leaf node?
print(f"Is End (root): {is_end}")

player = game.player(state)  # Whose turn is it?
print(f"Player (root): {player}")

successors = game.successors(state)  # For each action -> successor state
print(f"Successors (root): {successors}")

state = "A"
print(f"\nState: {state}")
is_end = game.is_end(state)
print(f"Is End (A): {is_end}")

player = game.player(state)
print(f"Player (A): {player}")

successors = game.successors(state)
print(f"Successors (A): {successors}")

state = "A1"
print(f"\nState: {state}")
is_end = game.is_end(state)
print(f"Is End (A1): {is_end}")

utility = game.utility(state)
print(f"Utility (A1): {utility}")

Start State: root
Is End (root): False
Player (root): agent
Successors (root): {'A': 'A', 'B': 'B', 'C': 'C'}

State: A
Is End (A): False
Player (A): opp
Successors (A): {'1': 'A1', '2': 'A2'}

State: A1
Is End (A1): True
Utility (A1): -50


### Havling Game

In [18]:
from dataclasses import dataclass
@dataclass(frozen=True)

class HalvingState:
    n: int
    player: str
    def __str__(self) -> str:
        return f"({self.n}, {self.player})"

class HalvingGame(Game):
    """
    Halving game:
    - Two players take turns halving or decrementing a number.
    - The player that is left with 0 wins.
    """
    def __init__(self, n: int):
        self.n = n
    def start_state(self) -> Any:
        return HalvingState(n=self.n, player="agent")

    def successors(self, state: HalvingState) -> dict[str, Any]:
        next_player = "opp" if state.player == "agent" else "agent"
        return {
            "decrement": HalvingState(n=state.n - 1, player=next_player),
            "half": HalvingState(n=state.n // 2, player=next_player),
        }

    def player(self, state: HalvingState) -> str:
        return state.player
    def is_end(self, state: HalvingState) -> bool:
        return state.n == 0
    def utility(self, state: HalvingState) -> float:
        assert state.n == 0
        if state.player == "agent":
            return +1  # Agent wins
        else:
            return -1  # Opponent wins

In [19]:
game = HalvingGame(n=11)

state = game.start_state()
print(f"state: {state}")

player = game.player(state)
print(f"player: {player}")

successors = game.successors(state)
print(f"successors: {successors}")

is_end = game.is_end(state)
print(f"is_end: {is_end}")

state: (11, agent)
player: agent
successors: {'decrement': HalvingState(n=10, player='opp'), 'half': HalvingState(n=5, player='opp')}
is_end: False


### Rollout

In [23]:
import numpy as np
import random

Policy = Callable[[Any], Any]
State = Any
Action = str

def set_random_seed(seed: int):
    """Set all random seeds for deterministic behavior."""
    random.seed(seed)
    np.random.seed(seed)

def sample_dict(choices: dict[Any, float]) -> Any:
    """Sample a key from a dictionary of choices based on their probabilities (values)."""
    return np.random.choice(list(choices.keys()), p=list(choices.values()))

@dataclass(frozen=True)
class Step:
    """Represents a step in a game (action and resulting state)."""
    action: Any
    state: Any
@dataclass(frozen=True)
class Rollout:
    """Represents a rollout of a game (sequence of actions that produces a utility)."""
    steps: list[Step]
    utility: float
def simulate(game: Game, policies: dict[str, Policy]) -> Rollout:
    """Simulate the game from the start state using the policy."""
    state = game.start_state()
    steps = []
    while not game.is_end(state):
        # Player whose turn it is chooses an action
        player = game.player(state)
        actions = policies[player](state)
        action = sample_dict(actions)
        # Advance the game state
        state = game.successors(state)[action]
        steps.append(Step(action=action, state=state))
    # See who wins?
    utility = game.utility(state)
    return Rollout(steps=steps, utility=utility)

In [24]:
# Agent always chooses A
def always_choose_a_policy(state: State) -> dict[Action, float]:
    return {"A": 1}
# Opponent chooses randomly between 1 and 2
def random_policy(state: State) -> dict[Action, float]:
    return {"1": 0.5, "2": 0.5}
policies = {
    "agent": always_choose_a_policy,
    "opp": random_policy,
}

In [31]:
game = Game1()
set_random_seed(1)
utility = simulate(game, policies)
print(f"Single simulation result: {utility}")

utilities = [simulate(game, policies).utility for _ in range(20)]
print(f"Utilities (20 runs): {utilities}")

mean_utility = np.mean(utilities)
print(f"Mean utility: {mean_utility}")


utilities = [simulate(game, policies).utility for _ in range(200)]
print(f"Utilities (2000 runs): {utilities}")

mean_utility = np.mean(utilities)
print(f"Mean utility: {mean_utility}")

utilities = [simulate(game, policies).utility for _ in range(20_000)]
print(f"Utilities (2000000 runs): {utilities}")

mean_utility = np.mean(utilities)
print(f"Mean utility: {mean_utility}")

Single simulation result: Rollout(steps=[Step(action=np.str_('A'), state='A'), Step(action=np.str_('2'), state='A2')], utility=50)
Utilities (20 runs): [-50, -50, -50, 50, 50, 50, 50, 50, -50, 50, 50, 50, -50, 50, -50, 50, -50, 50, 50, 50]
Mean utility: 15.0
Utilities (2000 runs): [50, -50, -50, -50, 50, -50, -50, -50, 50, -50, -50, 50, 50, 50, -50, 50, -50, -50, 50, 50, -50, 50, 50, 50, 50, 50, -50, 50, 50, 50, -50, 50, 50, 50, -50, 50, 50, 50, -50, -50, -50, 50, 50, -50, 50, -50, -50, -50, 50, -50, -50, -50, 50, -50, 50, -50, 50, 50, 50, -50, 50, -50, -50, 50, 50, -50, 50, -50, -50, -50, -50, 50, 50, 50, -50, 50, 50, -50, 50, 50, 50, -50, 50, 50, 50, -50, 50, 50, -50, -50, 50, 50, 50, 50, 50, -50, 50, 50, 50, 50, -50, 50, -50, 50, 50, 50, 50, 50, 50, 50, 50, -50, -50, -50, -50, -50, 50, -50, 50, 50, -50, 50, 50, 50, 50, -50, -50, -50, -50, 50, 50, -50, -50, -50, 50, -50, -50, -50, 50, 50, -50, -50, 50, 50, 50, 50, -50, 50, 50, 50, 50, 50, 50, -50, 50, -50, -50, -50, 50, -50, 50, 50, 

### Value Evaluation with Probability

In [32]:
def V_eval(game: Game, policies: dict[str, Policy], state: Any) -> float:
    """Return the value of the game."""
    # At the end of the game?
    if game.is_end(state):
        return game.utility(state)
    # Whose turn is it?
    player = game.player(state)
    policy = policies[player]
    # Try all actions
    value = 0
    for action, prob in policy(state).items():
        next_state = game.successors(state)[action]
        value += prob * V_eval(game, policies, next_state)
    return value

In [33]:
state = game.start_state()
value = V_eval(game, policies, state)

### Expectimax

In [ ]:
def random_policy(state: Any) -> Any:
    return {"1": 0.5, "2": 0.5}